# Text Classification
## Building  First Machine Learning Model to Classify Text
### Comparing TF-IDF vs Word Embeddings with Logistic Regression

##  Table of Contents

1. **What is Text Classification?** - Understanding the problem
2. **What is the 20 Newsgroups Dataset?** - Our data source
3. **The Two Approaches We'll Compare:**
   - TF-IDF (Term Frequency-Inverse Document Frequency)
   - Word Embeddings (GloVe vectors)
4. **Step-by-Step Implementation**
5. **Results & Comparison**
6. **Try It Yourself!**

---

## What is Text Classification?

**Text classification** is the task of automatically assigning a **category** (or "label") to a piece of text.

### Real-World Examples:

| Input Text | Task | Output |
|------------|------|--------|
| "I love this product!" | Sentiment Analysis | Positive ✅ |
| "Meeting at 3pm tomorrow" | Email Categorization | Calendar/Schedule 📅 |
| "How do I reset my password?" | Support Ticket Routing | Account Issues 🔑 |
| "Breaking: Stock market hits record high" | News Classification | Finance 📈 |

### In This Notebook:
We'll classify **newsgroup posts** into categories like:
- Sports (baseball, hockey)
- Science (medicine, space)

The computer will learn to read a post and decide: *"Is this about baseball, hockey, medicine, or space?"*






## What is the 20 Newsgroups Dataset?

The **20 Newsgroups dataset** is a classic dataset used for text classification research. It contains approximately **20,000 newsgroup posts** organized into **20 different topics**.

### What's a "Newsgroup"?

Before social media and Reddit, people used **newsgroups** (kind of like online forums) to discuss topics. Each newsgroup was dedicated to a specific subject:

- `rec.sport.baseball` → Baseball discussions
- `sci.space` → Space and astronomy discussions
- `comp.graphics` → Computer graphics discussions
- etc.

### Why Use This Dataset?

1. **It's built into scikit-learn** - Easy to load, no downloads needed!
2. **Real text data** - Messy, natural language (not artificially cleaned)
3. **Multi-class problem** - More interesting than just positive/negative
4. **Small enough to run quickly** - Great for learning!

### For This Tutorial:
We'll use **only 4 categories** to keep things simple and fast:
- 🏀 `rec.sport.baseball`
- 🏒 `rec.sport.hockey`  
- 🏥 `sci.med` (medicine)
- 🚀 `sci.space`


---

## The Machine Learning Pipeline (Big Picture)

Before we dive into code, let's understand the **overall process**:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        MACHINE LEARNING PIPELINE                             │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│   1. RAW TEXT          2. PREPROCESSING        3. FEATURE EXTRACTION       │
│   ─────────────        ───────────────         ──────────────────────      │
│   "The quick brown     "quick brown fox"       [0.2, 0.0, 0.8, ...]        │
│    fox jumps..."       (cleaned tokens)        (numbers!)                  │
│                                                                             │
│                                                                             │
│   4. TRAIN MODEL       5. MAKE PREDICTIONS     6. EVALUATE                 │
│   ──────────────       ──────────────────      ────────────                │
│   Learn patterns       New text → Category     How accurate?               │
│   from examples                                                            │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

### The Key Challenge: Computers Don't Understand Text!

Computers work with **numbers**, not words. So we need to convert text into numbers somehow. This is called **feature extraction** or **vectorization**.

**This notebook compares TWO different ways to convert text → numbers:**

1. **TF-IDF** (word frequency statistics)
2. **Word Embeddings** (learned semantic representations)

---

## The Two Approaches: TF-IDF vs Word Embeddings

### Approach 1: TF-IDF (Term Frequency-Inverse Document Frequency)

**What it does:** Converts each document into a vector where each dimension represents a word, and the value represents how "important" that word is to the document.

**Intuition:**
- Words that appear **frequently in a document** are probably important to that document
- BUT words that appear in **every document** (like "the", "is", "and") are not very informative
- TF-IDF balances these two ideas!

**Example:**
```
Document: "The NASA spacecraft landed on Mars"

TF-IDF might give:
  'nasa'      → 0.45  (important! specific to this doc)
  'spacecraft'→ 0.42  (important! specific)
  'mars'      → 0.38  (important! specific)
  'the'       → 0.01  (not important - appears everywhere)
  'on'        → 0.02  (not important - common word)
```

---

### Approach 2: Word Embeddings (GloVe)

**What it does:** Uses pre-trained vectors where each word is represented as a dense vector of ~100 numbers. Words with similar meanings have similar vectors.

**Intuition:**
- "king" and "queen" should have similar vectors (both royalty)
- "dog" and "puppy" should be close (both canines)
- "car" and "sandwich" should be far apart (unrelated)

**Example:**
```
'king'   → [0.2, -0.4, 0.8, 0.1, ...]  (100 numbers)
'queen'  → [0.3, -0.3, 0.7, 0.2, ...]  (similar to king!)
'apple'  → [-0.5, 0.2, 0.1, -0.8, ...] (very different)
```

For a **document**, we'll average all the word vectors together.

---

## Let's Start Coding!

Now that you understand the concepts, let's implement everything step by step.

**What we'll do:**
1. Install and import necessary libraries
2. Load the dataset and explore it
3. Preprocess the text (clean it up)
4. Build TF-IDF features and train a model
5. Build embedding features and train another model
6. Compare the results!

##  Setup: Installing Required Libraries

We need a few Python libraries:

| Library | Purpose |
|---------|--------|
| **scikit-learn** | Machine learning (already installed in Colab) |
| **spaCy** | Text preprocessing (tokenization, lemmatization) |
| **gensim** | Loading pre-trained word embeddings |
| **numpy** | Numerical operations (already installed) |

**Run the cell below to install what we need:**

---

## Imports: Loading Our Tools

Now we import all the libraries we'll use. Think of this like getting all your cooking ingredients ready before you start cooking!

**What each import does:**
- `numpy` - Math operations on arrays of numbers
- `sklearn` - Machine learning library (models, metrics, data)
- `gensim` - For loading word embeddings
- `spacy` - For text preprocessing


In [1]:
# Core python
import re
import numpy as np
from collections import Counter

# Data
from sklearn.datasets import fetch_20newsgroups

# Model+ evaluation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

#TF-IDF features
from sklearn.feature_extraction.text import TfidfVectorizer

# Embedding 
import gensim.downloader as api

# preprpcessing
import spacy

# Make results reproducible 
RANDOM_SEED =42
np.random.seed(RANDOM_SEED)



---

## Load and Explore Data

Now we load our dataset! This is always the first real step in any ML project.

### What we're loading:
We will load **only 4 categories** to keep things simple:

| Category | What it contains |
|----------|------------------|
| 🏀 `rec.sport.baseball` | Baseball discussions, scores, players |
| 🏒 `rec.sport.hockey` | Hockey discussions, NHL, players |
| 🏥 `sci.med` | Medical topics, health, diseases |
| 🚀 `sci.space` | Space exploration, NASA, astronomy |

**Why only 4?** Using fewer categories makes the notebook run faster and easier to understand. The same techniques work for all 20 categories!